<a href="https://colab.research.google.com/github/juniti-y/RLseminar/blob/main/RLex02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# モジュールの呼び出し

In [1]:
# numpyモジュールの呼び出し
import numpy as np

# matplotlibモジュールの呼び出し (%matplotlib行は環境に応じて変更すること)
import matplotlib.pyplot as plt

# 環境クラスを生成

## T迷路環境をクラスとして定義

In [2]:
class Tmaze():
    """
    T迷路の環境を定義するクラス
    [メンバー変数]
    - num_state: 状態の総数
    - num_action: 行動の総数
    - state: 各時刻で訪問している状態のインデクス
      - state = 0 -> J1
      - state = 1 -> J2
      - state = 2 -> J3
      - state = 3 -> G
      にそれぞれ対応
    """
    # 状態の総数を表す定数
    num_state = 4
    # 行動の総数を表す定数
    num_action = 2
    # 現在の状態を表す変数
    state = 0

    def __init__(self):
        """
        コンストラクタ
        """
        pass

    def show_state(self):
        """
        現在の状態を標準出力するメソッド関数
        """
        print(self.state)

    def get_env_config(self):
        """
        環境設定（状態数, 行動数）を取得するためのメソッド関数
        """
        return self.num_state, self.num_action

    def init_state(self):
        """
        状態を初期化（初期状態に戻す）メソッド関数
        """
        self.state = 0
        reward = None
        return reward, self.state

    # 選択した行動に従って状態を遷移させるメソッド関数
    # action = 0: 左
    # action = 1: 右
    def move_state(self, action):
        """
        選択した行動 action に従って状態を遷移させるメソッド関数
        [入力引数]
        - action: 行動のインデクス
          - action = 0: 左
          - action = 1: 右
        """
        if self.state == 0:
            if action == 0:
                self.state = 1
                reward = -1.0
            else:
                self.state = 2
                reward = 0.0
        elif self.state == 1:
            if action == 0:
                self.state = 3
                reward = 10.0
            else:
                self.state = 3
                reward = 0.0
        elif self.state ==2:
            if action == 0:
                self.state = 3
                reward = 2.0
            else:
                self.state = 3
                reward = -10.0
        return reward, self.state

    def reach_goal(self):
        """
        ゴールに到達したか否かを伝えるメソッド関数
        [返り値]
            - True: ゴールに到達
            - False: ゴールに未到達
        """
        if self.state == 3:
            return True
        else:
            return False

## ダンジョン環境をクラスとして定義

In [3]:
class Dungeon(Tmaze):
    def __init__(self, config_file):
        """
        コンストラクタ
        """
        # CSV形式で保存されているマップ情報を表示する
        self.map_info = np.loadtxt(config_file, delimiter=',', dtype = 'int64')
        # マップのサイズを読み取る
        [self.vsize, self.hsize] =  self.map_info.shape
        # 状態の総数を表す定数
        self.num_state = self.vsize * self.hsize
        # 行動の総数を表す定数
        self.num_action = 4
        # マップ情報から初期状態位置を取得を取得する
        match_idx = np.where(self.map_info.ravel() == 2)
        self.state0 = match_idx[0][0]
        # マップ情報からゴール状態位置を取得を取得する
        match_idx = np.where(self.map_info.ravel() == 3)
        self.stateG = match_idx[0][0]

        # 現在の状態を仮初期化
        self.init_state()

    def init_state(self):
        """
        状態を初期化（初期状態に戻す）メソッド関数
        """
        # 現在の状態を初期状態に変更する
        self.state = self.state0
        # 状態番号からグリッド上の横位置へと変換
        self.h_loc = self.state % self.hsize
        # 状態番号からグリッド上の縦位置へと変換
        self.v_loc = self.state // self.hsize
        # 報酬を欠損値にしておく
        reward = None

        return reward, self.state

    def move_state(self, action):
        """
        選択した行動 action に従って状態を遷移させるメソッド関数
        [入力引数]
        - action: 行動のインデクス
          - action = 0: 上
          - action = 1: 右
          - action = 2: 下
          - action = 3: 左
        """
        if action == 0:
            ii = self.v_loc - 1
            jj = self.h_loc
            if ii < 0:
                ii = ii + 1
            elif self.map_info[ii][jj] == 0:
                ii = ii + 1
        elif action == 1:
            ii = self.v_loc
            jj = self.h_loc + 1
            if jj == self.hsize:
                jj = jj -1
            elif self.map_info[ii][jj] == 0:
                jj = jj -1
        elif action == 2:
            ii = self.v_loc + 1
            jj = self.h_loc
            if ii == self.vsize:
                ii = ii - 1
            elif self.map_info[ii][jj] == 0:
                ii = ii - 1
        elif action == 3:
            ii = self.v_loc
            jj = self.h_loc - 1
            if jj < 0:
                jj = jj + 1
            elif self.map_info[ii][jj] == 0:
                jj = jj + 1
        self.state = ii * self.hsize + jj
        self.v_loc = ii
        self.h_loc = jj

        if self.reach_goal():
            reward = 0
        else:
            reward = -1

        return reward, self.state


    def reach_goal(self):
        """
        ゴールに到達したか否かを伝えるメソッド関数
        [返り値]
            - True: ゴールに到達
            - False: ゴールに未到達
        """
        if self.state == self.stateG:
            return True
        else:
            return False

    def print_map_info(self):
        """
        mapの情報を標準出力する
        """
        A = self.map_info.copy()
        A[self.v_loc][self.h_loc] = 8
        print(A)
        return

    def draw_map_info(self, fig):
        """
        mapの情報をグリッド状に表示する
        """
        A = self.map_info.copy()
        A[self.v_loc][self.h_loc] = 8
        fig.clear()
        ax1 = fig.add_subplot(1,1,1)
        ax1.imshow(A, interpolation='nearest', vmin=0, vmax=8, cmap='jet')
        plt.pause(0.02)
        plt.show()
        return

    def draw_value_info(self, fig, Q):
        """
        価値関数の情報をmap上にスーパーインポーズ
        """
        A = np.max(agent1.Q, axis=1)
        match_idx = np.where(self.map_info.ravel() == 0)
        A[match_idx] = np.nan
        B = A.reshape([self.vsize, self.hsize])

        fig.clear()
        ax1 = fig.add_subplot(1,1,1)
        ax1.imshow(B, interpolation='nearest', cmap='jet')
        plt.pause(0.02)
        plt.show()
        return